# Notebook 2: Geographic & Cross-Sectional Patterns

Where are pension funds located, how do they differ by state/region, and which systems are outliers?

**Sections:**
1. Coverage by state and Census region
2. Regional allocation profiles (2001 vs. 2021)
3. Omega by region (2021)
4. State-level leaders & laggards
5. Funding ratio geography
6. Outlier systems
7. Size effects (AUM vs. allocation)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'blue': '#1f77b4', 'orange': '#d35400', 'purple': '#7f6d9e',
    'light_blue': '#aec7e8', 'red': '#d62728', 'green': '#2ca02c',
}

OUTPUT_DIR = Path('../../output_python/exploration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load cleaned data
df = pd.read_csv('../../output_python/ppd_plan_level_clean.csv')

# Construct derived variables (same as replication notebook)
df['real_assets_trgt'] = df['RETotal_Trgt'].fillna(0) + df['COMDTotal_Trgt'].fillna(0)
df['alternatives_trgt'] = (
    df['PETotal_Trgt'].fillna(0) + df['real_assets_trgt']
    + df['HFTotal_Trgt'].fillna(0) + df['AltMiscTotal_Trgt'].fillna(0)
    + df['OtherTotal_Trgt'].fillna(0)
)
df['public_equity_trgt'] = df['EQTotal_Trgt'].fillna(0)
df['fixed_income_trgt'] = df['FITotal_Trgt'].fillna(0) + df['CashTotal_Trgt'].fillna(0)
df['risky_trgt'] = df['public_equity_trgt'] + df['alternatives_trgt']

denom = df['alternatives_trgt'] + df['public_equity_trgt']
df['omega'] = np.where(denom > 0, df['alternatives_trgt'] / denom, np.nan)

# System-level aggregation (AUM-weighted)
def weighted_avg(group, col, weight_col='MktAssets_net'):
    mask = group[col].notna() & group[weight_col].notna() & (group[weight_col] > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(group.loc[mask, col], weights=group.loc[mask, weight_col])

share_cols = [
    'alternatives_trgt', 'public_equity_trgt', 'fixed_income_trgt', 'risky_trgt',
    'EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
    'HFTotal_Trgt', 'CashTotal_Trgt', 'AltMiscTotal_Trgt',
    'omega', 'ActFundedRatio_GASB', 'InvestmentReturnAssumption_GASB',
    'InvestmentReturn_1yr', 'ret_bgnassets',
]
sum_cols = ['MktAssets_net', 'TotMembership', 'beneficiaries_tot', 'actives_tot']
first_cols = ['pub_system_name', 'StateAbbrev', 'region', 'division', 'general_consultant']

rows = []
for (pub_id, fy), grp in df.groupby(['pub_id', 'fy']):
    row = {'pub_id': pub_id, 'fy': fy}
    for col in share_cols:
        if col in grp.columns:
            row[col] = weighted_avg(grp, col)
    for col in sum_cols:
        if col in grp.columns:
            row[col] = grp[col].sum()
    for col in first_cols:
        if col in grp.columns:
            row[col] = grp[col].iloc[0]
    rows.append(row)

sys_df = pd.DataFrame(rows)
print(f'System-level: {sys_df.shape[0]} obs, {sys_df["pub_id"].nunique()} systems')
print(f'FY range: {sys_df["fy"].min():.0f}–{sys_df["fy"].max():.0f}')

## Section 1: Coverage by State and Census Region

In [ ]:
# Count systems by state (using latest year)
latest_yr = int(sys_df['fy'].max())
sys_latest = sys_df[sys_df['fy'] == latest_yr].copy()

state_counts = sys_latest.groupby('StateAbbrev').agg(
    n_systems=('pub_id', 'nunique'),
    total_aum=('MktAssets_net', 'sum'),
    region=('region', 'first'),
).sort_values('n_systems', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Systems by state
ax = axes[0]
top_states = state_counts.head(25)
colors_by_region = {'Northeast': '#1f77b4', 'South': '#d35400', 'Midwest': '#2ca02c', 'West': '#7f6d9e'}
bar_colors = [colors_by_region.get(r, 'gray') for r in top_states['region']]
ax.barh(range(len(top_states)), top_states['n_systems'], color=bar_colors, alpha=0.85)
ax.set_yticks(range(len(top_states)))
ax.set_yticklabels(top_states.index, fontsize=9)
ax.set_xlabel('Number of Systems')
ax.set_title(f'Pension Systems by State (FY{latest_yr}, top 25)')
ax.invert_yaxis()
# Legend
for region, color in colors_by_region.items():
    ax.bar(0, 0, color=color, label=region)
ax.legend(loc='lower right', fontsize=9)

# Right: Systems by Census region
ax = axes[1]
region_counts = sys_latest.groupby('region')['pub_id'].nunique().sort_values(ascending=False)
region_aum = sys_latest.groupby('region')['MktAssets_net'].sum().sort_values(ascending=False)
x = np.arange(len(region_counts))
w = 0.35
ax.bar(x - w/2, region_counts.values, w, color=COLORS['blue'], label='# Systems', alpha=0.85)
ax2 = ax.twinx()
ax2.bar(x + w/2, region_aum.values / 1e12, w, color=COLORS['orange'], label='AUM ($T)', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(region_counts.index, fontsize=10)
ax.set_ylabel('Number of Systems')
ax2.set_ylabel('Total AUM ($ Trillions)')
ax.set_title(f'Systems & AUM by Census Region (FY{latest_yr})')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_coverage.png')
plt.show()

print(f'States represented: {sys_latest["StateAbbrev"].nunique()}')
print(f'Regions: {sys_latest["region"].value_counts().to_dict()}')

## Section 2: Regional Allocation Profiles (2001 vs. 2021)

AUM-weighted average allocation by Census region — how did the shift to alternatives vary geographically?

In [ ]:
# Regional allocation profiles: 2001 vs 2021
alloc_cols = ['public_equity_trgt', 'fixed_income_trgt', 'alternatives_trgt']
alloc_labels = ['Public Equity', 'Fixed Income', 'Alternatives']
alloc_colors = [COLORS['blue'], COLORS['green'], COLORS['orange']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, yr in enumerate([2001, 2021]):
    ax = axes[idx]
    sub = sys_df[sys_df['fy'] == yr].dropna(subset=['region'])
    
    # AUM-weighted averages by region
    region_alloc = []
    for region in ['Northeast', 'Midwest', 'South', 'West']:
        rsub = sub[sub['region'] == region]
        if len(rsub) == 0:
            continue
        row = {'region': region}
        for col in alloc_cols:
            mask = rsub[col].notna() & rsub['MktAssets_net'].notna() & (rsub['MktAssets_net'] > 0)
            if mask.sum() > 0:
                row[col] = np.average(rsub.loc[mask, col], weights=rsub.loc[mask, 'MktAssets_net']) * 100
            else:
                row[col] = 0
        region_alloc.append(row)
    
    ra_df = pd.DataFrame(region_alloc).set_index('region')
    
    bottom = np.zeros(len(ra_df))
    for col, label, color in zip(alloc_cols, alloc_labels, alloc_colors):
        ax.bar(ra_df.index, ra_df[col], bottom=bottom, label=label, color=color, alpha=0.85)
        # Add value labels
        for j, (val, b) in enumerate(zip(ra_df[col], bottom)):
            if val > 3:
                ax.text(j, b + val/2, f'{val:.0f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        bottom += ra_df[col].values
    
    ax.set_ylabel('Target Allocation (%)')
    ax.set_title(f'FY{yr}')
    ax.set_ylim(0, 105)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9)

fig.suptitle('Asset Allocation by Census Region', fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_regional_allocation.png')
plt.show()

## Section 3: Omega by Region (2021)

Which regions adopted alternatives fastest?

In [ ]:
# Box plots of omega by Census region for 2021
sys_2021 = sys_df[sys_df['fy'] == 2021].dropna(subset=['omega', 'region']).copy()
sys_2021['omega_pct'] = sys_2021['omega'] * 100

# Order regions by median omega
region_order = sys_2021.groupby('region')['omega_pct'].median().sort_values(ascending=False).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Box plot
ax = axes[0]
sns.boxplot(data=sys_2021, x='region', y='omega_pct', order=region_order,
            palette=[colors_by_region.get(r, 'gray') for r in region_order],
            ax=ax, fliersize=3)
ax.set_xlabel('')
ax.set_ylabel('Omega (Alt-to-Risky Share, %)')
ax.set_title('Omega by Census Region (FY2021)')
ax.grid(axis='y', alpha=0.3)

# Right: Box plot by division (more granular)
sys_2021_div = sys_2021.dropna(subset=['division'])
div_order = sys_2021_div.groupby('division')['omega_pct'].median().sort_values(ascending=False).index.tolist()
ax = axes[1]
sns.boxplot(data=sys_2021_div, x='division', y='omega_pct', order=div_order,
            color=COLORS['light_blue'], ax=ax, fliersize=3)
ax.set_xlabel('')
ax.set_ylabel('Omega (%)')
ax.set_title('Omega by Census Division (FY2021)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_omega_by_region.png')
plt.show()

# Print regional medians
print('Median omega by region (FY2021):')
for r in region_order:
    med = sys_2021.loc[sys_2021['region'] == r, 'omega_pct'].median()
    n = sys_2021.loc[sys_2021['region'] == r].shape[0]
    print(f'  {r}: {med:.1f}% (N={n})')

## Section 4: State-Level Leaders & Laggards

Rank states by mean omega in 2021.

In [ ]:
# State-level omega ranking (AUM-weighted, 2021)
state_omega = []
for state in sys_2021['StateAbbrev'].dropna().unique():
    ssub = sys_2021[sys_2021['StateAbbrev'] == state]
    mask = ssub['omega'].notna() & ssub['MktAssets_net'].notna() & (ssub['MktAssets_net'] > 0)
    if mask.sum() == 0:
        continue
    omega_w = np.average(ssub.loc[mask, 'omega'], weights=ssub.loc[mask, 'MktAssets_net']) * 100
    aum = ssub['MktAssets_net'].sum() / 1e9
    region = ssub['region'].iloc[0]
    state_omega.append({'state': state, 'omega': omega_w, 'aum_B': aum, 'region': region, 'n_systems': len(ssub)})

state_omega = pd.DataFrame(state_omega).sort_values('omega', ascending=True)

fig, ax = plt.subplots(figsize=(10, 14))
bar_colors = [colors_by_region.get(r, 'gray') for r in state_omega['region']]
ax.barh(range(len(state_omega)), state_omega['omega'], color=bar_colors, alpha=0.85)
ax.set_yticks(range(len(state_omega)))
ax.set_yticklabels(state_omega['state'], fontsize=8)
ax.set_xlabel('AUM-Weighted Omega (%)')
ax.set_title('Alternative-to-Risky Share by State (FY2021)')
ax.axvline(sys_2021['omega_pct'].median(), color='red', ls='--', alpha=0.5, label=f'Median: {sys_2021["omega_pct"].median():.0f}%')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Add region legend
for region, color in colors_by_region.items():
    ax.barh(0, 0, color=color, label=region)
ax.legend(loc='lower right', fontsize=9)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_state_omega_ranking.png')
plt.show()

# Print top 5 and bottom 5
print('Top 5 states by omega (FY2021):')
for _, r in state_omega.tail(5).iloc[::-1].iterrows():
    print(f'  {r["state"]}: {r["omega"]:.1f}% (AUM: ${r["aum_B"]:.1f}B, {r["region"]})')
print('\nBottom 5 states by omega (FY2021):')
for _, r in state_omega.head(5).iterrows():
    print(f'  {r["state"]}: {r["omega"]:.1f}% (AUM: ${r["aum_B"]:.1f}B, {r["region"]})')

## Section 5: Funding Ratio Geography

Do underfunded states take more risk?

In [ ]:
# Funding ratio vs. omega by state (2021) — scatter plot
state_fund = []
for state in sys_2021['StateAbbrev'].dropna().unique():
    ssub = sys_2021[sys_2021['StateAbbrev'] == state]
    mask_o = ssub['omega'].notna() & ssub['MktAssets_net'].notna() & (ssub['MktAssets_net'] > 0)
    mask_f = ssub['ActFundedRatio_GASB'].notna() & ssub['MktAssets_net'].notna() & (ssub['MktAssets_net'] > 0)
    if mask_o.sum() == 0 or mask_f.sum() == 0:
        continue
    omega_w = np.average(ssub.loc[mask_o, 'omega'], weights=ssub.loc[mask_o, 'MktAssets_net']) * 100
    fund_w = np.average(ssub.loc[mask_f, 'ActFundedRatio_GASB'], weights=ssub.loc[mask_f, 'MktAssets_net']) * 100
    aum = ssub['MktAssets_net'].sum() / 1e9
    region = ssub['region'].iloc[0]
    state_fund.append({'state': state, 'omega': omega_w, 'funding': fund_w, 'aum_B': aum, 'region': region})

sf_df = pd.DataFrame(state_fund)

fig, ax = plt.subplots(figsize=(10, 7))
for region, color in colors_by_region.items():
    mask = sf_df['region'] == region
    ax.scatter(sf_df.loc[mask, 'funding'], sf_df.loc[mask, 'omega'],
               s=sf_df.loc[mask, 'aum_B'] * 2, color=color, alpha=0.7,
               edgecolor='white', label=region)
    for _, r in sf_df[mask].iterrows():
        ax.annotate(r['state'], (r['funding'], r['omega']), fontsize=7, alpha=0.8)

# OLS fit line
X = sm.add_constant(sf_df['funding'])
model = sm.OLS(sf_df['omega'], X).fit()
x_line = np.linspace(sf_df['funding'].min(), sf_df['funding'].max(), 100)
ax.plot(x_line, model.params['const'] + model.params['funding'] * x_line,
        '--', color='red', alpha=0.6, lw=1.5)

ax.set_xlabel('AUM-Weighted GASB Funding Ratio (%)')
ax.set_ylabel('AUM-Weighted Omega (%)')
ax.set_title(f'Funding Ratio vs. Alternatives Adoption by State (FY2021)\n'
             f'OLS: beta={model.params["funding"]:.3f}, t={model.tvalues["funding"]:.2f}, R²={model.rsquared:.2f}')
ax.legend(title='Region', markerscale=0.7)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_funding_vs_omega.png')
plt.show()

## Section 6: Outlier Systems

Profile the most extreme pension systems across multiple dimensions.

In [ ]:
# Outlier identification (FY2021 system-level)
sys_2021_full = sys_df[sys_df['fy'] == 2021].copy()

categories = [
    ('Highest Omega (Alternatives Share)', 'omega', False),
    ('Lowest Omega', 'omega', True),
    ('Largest AUM', 'MktAssets_net', False),
    ('Lowest Funding Ratio', 'ActFundedRatio_GASB', True),
    ('Highest Funding Ratio', 'ActFundedRatio_GASB', False),
]

for title, col, ascending in categories:
    sub = sys_2021_full.dropna(subset=[col]).sort_values(col, ascending=ascending).head(5)
    print(f'\n{title} (FY2021):')
    print('-' * 80)
    for _, r in sub.iterrows():
        omega_str = f'{r["omega"]*100:.1f}%' if pd.notna(r.get('omega')) else 'N/A'
        fund_str = f'{r["ActFundedRatio_GASB"]*100:.0f}%' if pd.notna(r.get('ActFundedRatio_GASB')) else 'N/A'
        aum_str = f'${r["MktAssets_net"]/1e9:.1f}B' if pd.notna(r.get('MktAssets_net')) else 'N/A'
        print(f'  {r["pub_system_name"]:45s} | {r["StateAbbrev"]:2s} | '
              f'Omega={omega_str:>6s} | Funding={fund_str:>5s} | AUM={aum_str:>8s}')

## Section 7: Size Effects

Do larger pension systems allocate differently?

In [ ]:
# Binscatter: log(AUM) vs omega and funding ratio (2021)
def binscatter(x, y, n_bins=20, ax=None, xlabel='', ylabel='', title='',
               color=None):
    data = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(data) < n_bins:
        n_bins = max(len(data) // 2, 2)
    data['bin'] = pd.qcut(data['x'], n_bins, labels=False, duplicates='drop')
    binned = data.groupby('bin')[['x', 'y']].mean()

    X_ols = sm.add_constant(data['x'])
    model = sm.OLS(data['y'], X_ols).fit(cov_type='HC1')
    beta = model.params['x']
    tstat = model.tvalues['x']
    r2 = model.rsquared
    n = int(model.nobs)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    c = color or COLORS['blue']
    ax.scatter(binned['x'], binned['y'], s=60, color=c, zorder=5)
    x_line = np.linspace(data['x'].min(), data['x'].max(), 100)
    y_line = model.params['const'] + beta * x_line
    ax.plot(x_line, y_line, '--', color=COLORS['red'], lw=1.5, alpha=0.8)
    ax.text(0.05, 0.95, f'beta={beta:.3f}\nt={tstat:.2f}\nR²={r2:.2f}\nN={n}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    return model

sys_2021_size = sys_2021_full.dropna(subset=['MktAssets_net', 'omega']).copy()
sys_2021_size = sys_2021_size[sys_2021_size['MktAssets_net'] > 0]
sys_2021_size['log_aum'] = np.log10(sys_2021_size['MktAssets_net'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

binscatter(sys_2021_size['log_aum'], sys_2021_size['omega'] * 100,
           ax=axes[0],
           xlabel='log10(AUM)', ylabel='Omega (%)',
           title='AUM vs. Alt-to-Risky Share (FY2021)')

sys_2021_fund = sys_2021_full.dropna(subset=['MktAssets_net', 'ActFundedRatio_GASB']).copy()
sys_2021_fund = sys_2021_fund[sys_2021_fund['MktAssets_net'] > 0]
sys_2021_fund['log_aum'] = np.log10(sys_2021_fund['MktAssets_net'])

binscatter(sys_2021_fund['log_aum'], sys_2021_fund['ActFundedRatio_GASB'] * 100,
           ax=axes[1],
           xlabel='log10(AUM)', ylabel='GASB Funding Ratio (%)',
           title='AUM vs. Funding Ratio (FY2021)')

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_size_effects.png')
plt.show()